<a href="https://colab.research.google.com/github/siya-pathak/from-scratch/blob/main/gpt2_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-10-02 10:02:12--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-10-02 10:02:13 (22.6 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [5]:
print("length of dataset in characters: ", len(text))
print(text[:300])

length of dataset in characters:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


In [9]:
chars = sorted(list(set(text)))
print(''.join(chars))
print(len(chars)) #size of our vocabulary


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [13]:
#we want to tokenize here
#character level tokenization
stoi = {chi:i for i, chi in enumerate(chars)}
itos = {i:chi for i, chi in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
print(encode("shakespeare"))
print(decode(encode("shakespeare")))

[57, 46, 39, 49, 43, 57, 54, 43, 39, 56, 43]
shakespeare


In [20]:
#tokenize the entire dataset we have
#we use pytorch to make the vocab into a tensor
import torch
data =torch.tensor(encode(text))
print(data.shape, data.dtype)
print(data[:50])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56])


In [21]:
n = int(0.9*(len(data)))
train = data[:n]
val = data[n:]

In [22]:
#we do not simply feed in entire data to transformer -> compuatationally expensive
#we sample random data, chunks

In [24]:
context_len = 10
train[:context_len+1] #this has 10 examples packed in the sense of -> for these input, this is the char that would come next

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64])

In [26]:
x = train[:context_len]
y = train[1:context_len+1] #offset by 1
for t in range(context_len):
  context = x[:t+1]
  target = y[t]
  print(f"when input is {context} the target: {target}") #transformer should be in the habit of seeing as little as 1 charcater to block_size

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47]) the target: 64


In [32]:
#mini batches of multiple chunks of text, stacked up, for efficiency to keep GPUs busy
torch.manual_seed(28309123719)
batch_size = 4 #how many chunks are to be processed in parallel
context_len = 10 #characters to be processed for next prediction

def get_batch(split):
  data = train if split == "train" else val
  ix = torch.randint(len(data) - context_len, (batch_size,)) #this prodcues batch_size number of indexes (4 index because we need to send 4 chunks)
  x = torch.stack([data[i:i+context_len] for i in ix])
  y = torch.stack([data[i+1: i+context_len+1] for i in ix]) #torch.randint's upper bound is exclusive
  #if len(data) = 20 and context len is 10 max we can get is between 0 and 9 so if i = 9, we index 9 to 9 + 10 which is 10 numbers and then we do 10 to 20
  return x, y

xb, yb = get_batch("train")
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)
#the row vectors are the chunks we send for training, 4 rows of 10 columns

inputs:
torch.Size([4, 10])
tensor([[51, 40, 50, 43, 57, 10,  1, 39, 58,  1],
        [58,  6,  1, 44, 56, 43, 43, 50, 63,  0],
        [41, 58,  1, 53, 44,  1, 50, 53, 60, 43],
        [ 0, 31, 58, 39, 63,  1, 52, 53, 58,  1]])
targets:
torch.Size([4, 10])
tensor([[40, 50, 43, 57, 10,  1, 39, 58,  1, 57],
        [ 6,  1, 44, 56, 43, 43, 50, 63,  0, 32],
        [58,  1, 53, 44,  1, 50, 53, 60, 43,  1],
        [31, 58, 39, 63,  1, 52, 53, 58,  1, 58]])


In [33]:
for b in range(batch_size): # batch dimension - 4
    for t in range(context_len): # time dimension - 10
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

when input is [51] the target: 40
when input is [51, 40] the target: 50
when input is [51, 40, 50] the target: 43
when input is [51, 40, 50, 43] the target: 57
when input is [51, 40, 50, 43, 57] the target: 10
when input is [51, 40, 50, 43, 57, 10] the target: 1
when input is [51, 40, 50, 43, 57, 10, 1] the target: 39
when input is [51, 40, 50, 43, 57, 10, 1, 39] the target: 58
when input is [51, 40, 50, 43, 57, 10, 1, 39, 58] the target: 1
when input is [51, 40, 50, 43, 57, 10, 1, 39, 58, 1] the target: 57
when input is [58] the target: 6
when input is [58, 6] the target: 1
when input is [58, 6, 1] the target: 44
when input is [58, 6, 1, 44] the target: 56
when input is [58, 6, 1, 44, 56] the target: 43
when input is [58, 6, 1, 44, 56, 43] the target: 43
when input is [58, 6, 1, 44, 56, 43, 43] the target: 50
when input is [58, 6, 1, 44, 56, 43, 43, 50] the target: 63
when input is [58, 6, 1, 44, 56, 43, 43, 50, 63] the target: 0
when input is [58, 6, 1, 44, 56, 43, 43, 50, 63, 0] the

In [37]:
print(xb)
print(yb)

tensor([[51, 40, 50, 43, 57, 10,  1, 39, 58,  1],
        [58,  6,  1, 44, 56, 43, 43, 50, 63,  0],
        [41, 58,  1, 53, 44,  1, 50, 53, 60, 43],
        [ 0, 31, 58, 39, 63,  1, 52, 53, 58,  1]])
tensor([[40, 50, 43, 57, 10,  1, 39, 58,  1, 57],
        [ 6,  1, 44, 56, 43, 43, 50, 63,  0, 32],
        [58,  1, 53, 44,  1, 50, 53, 60, 43,  1],
        [31, 58, 39, 63,  1, 52, 53, 58,  1, 58]])


In [44]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(28309123719)

class BigramLanguageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
  # each token directly reads off the logits for the next token from a lookup table
    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

  def forward(self, idx, targets=None):
    #idx, targets or xb and yb are tensors of tensors of integers -> 2d
    logits = self.token_embedding_table(idx) #embeddings of the next token being any of the 65 token in vocab this probability is in the 3rd dimension
    print(logits.shape)
    B,T,C = logits.shape
    logits = logits.view(B*T, C)
    targets = targets.view(B*T)
    loss = F.cross_entropy(logits, targets)
    #loss = -log(p(y)) = -log(1/65) #currently all at 1/65
    return logits, loss

  def generate
m = BigramLanguageModel(vocab_size=65)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

torch.Size([4, 10, 65])
torch.Size([40, 65])
tensor(4.7721, grad_fn=<NllLossBackward0>)
